In [ ]:
using CairoMakie;       #Visualización en 3D
using LaTeXStrings;     #Paquetería para utilizar texto en LaTeX en las gráficas
using DelimitedFiles;   #Paquetería para leer y escribir archivos
using Printf;           #Paquetería para modificar el formato de las etiquetas en los ticks

DataPath = "Quasiperiodic-Tiles/Global Structural Studies/Data/SI_Fig4_SigmaSquare_gR/"; #Ruta donde se encuentran almacenados los datos

#Función para la longitud de escala λ_N 
AproxLambda(NSides) = (2π / (1 - cos(2π / NSides)));

In [ ]:
#Diccionario con los valores de la densidad numérica de los sistemas cuasiperiódicos en 2D con decoración en vértices
Rho_Dict = Dict(
                5  => 1.2328979808609704,
                7  => 1.2517957581185175,
                9  => 1.260284085456272,
                11 => 1.2645739922427748,
                13 => 1.2670366156186388,
                15 => 1.2685820147323164,
                17 => 1.2696141740269873,
                19 => 1.2703373895977548,
                21 => 1.2708642307351703,
                23 => 1.2712590235307708,
                25 => 1.271563821555834,
                27 => 1.271802352106815,
                29 => 1.2719940413420914,
                31 => 1.2721488757491926
               );

#Diccionario con los valores del offset vertical para las g(R) por grupos de prototeselas en un escalado Lin-Lin
YOffSet_Dict = Dict(
                    5  => 475,
                    7  => 420,
                    9  => 200,
                    11 => 100,
                    13 => 80,
                    15 => 90,
                    17 => 60,
                    19 => 50
                   );

### Visualización de la hiperuniformidad a partir de la $N(R)$

In [ ]:
###################################################################################################################
#                                   Datos del sistema cuasiperiódico 2D-1D
###################################################################################################################
NSides = 5;         #Simetría rotacional del sistema cuasiperiódico en 2D (Antes de la proyección)
Tamaño_Tesela = 1;  #Ranking de la tesela (1 = Más pequeña)
Radio = 500;        #Radio de la "vecindad circular" en 1D (Mitad del tamaño de la región cuadrada centrada en el origen)
###################################################################################################################
#                             Datos de los notebooks para la generación de datos
###################################################################################################################
Notebooks = 20;     #Número de Notebooks empleados
Vecindades = 500;   #Número de Vecindades por Notebook
###################################################################################################################
#                                   Datos para el cálculo de la N(R) y N^2(R)
###################################################################################################################
Steps = Int(1e5);       #Número de pasos realizados desde R = 0 hasta R = Radio
ΔR = Radio / Steps;     #Tamaño del salto entre R y R

for NSides in 5:2:17
    ###################################################################################################################
    #                                       Factor de normalización de Torquato
    ###################################################################################################################
    Rango = 0.0:ΔR:Radio;   #Intervalo de radios empleados
    Rho = Rho_Dict[NSides]; #Densidad del decorado en vértices, no en centroides
    FN = 2*sqrt(π*Rho);     #Factor de normalización para mantener los resultados independientes de la densidad de puntos
    Rango = FN .* Rango;    #Normalizamos las distancias por el factor de normalización de Torquato
    ###################################################################################################################
    #                                   Definición de las características del lienzo
    ###################################################################################################################
    Radio_Viz = 275;
    if NSides >= 5
        λ = AproxLambda(NSides);            #Longitud de escala del primer pico
        Radio_Viz = Int(ceil(10.2 * λ));
    end

    Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
    Sigma2_Ax = Axis(
                     Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                     title = L"N = %$(NSides)",                                   #Título de la gráfica
                     xlabel = L"R",                                               #Etiqueta que aparece en el eje horizontal
                     ylabel = L"\sigma^{2}(R) / R",                               #Etiqueta que aparece en el eje vertical
                     titlesize = 75,                                              #Tamaño del título
                     xlabelsize = 75,                                             #Tamaño de la etiqueta al eje horizontal
                     ylabelsize = 75,                                             #Tamaño de la etiqueta al eje vertical
                     xticklabelsize = 60,                                         #Tamaño para el eje X
                     yticklabelsize = 60,                                         #Tamaño para el eje Y
                     xticksize = 45,                                              #Tamaño de los ticks horizontales
                     yticksize = 45,                                              #Tamaño de los ticks verticales
                     limits = ((ΔR, Radio_Viz), nothing),                         #Límites de la visualización para la gráfica
                     ytickformat = values -> [@sprintf("%.1f", v) for v in values],
                    )
    hidespines!(Sigma2_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
    hidedecorations!(
                     Sigma2_Ax,
                     label = false,           #Se oculta o no las etiquetas a los ejes
                     ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                     ticks = false            #Se oculta o no los ticks de los ejes
                    )
    for Tamaño_Tesela in 1:Int(floor(NSides/2))
        Paleta_Colores = cgrad(:inferno, Int(floor(NSides/2)) + 1, categorical = true); #Definimos el gradiente de colores para la gráfica
        ###################################################################################################################
        #                                   Lectura de los datos σ^2(R)
        ###################################################################################################################
        σ2 = vec(readdlm(DataPath * "N$(NSides)/A$(Tamaño_Tesela)/Torquato_NR_N$(NSides)_Alfa0P0_R$(Radio)_Step1e5_Area$(Tamaño_Tesela)_SigmaCuadrada.csv"));
        ###################################################################################################################
        #                                            Gráfica de la σ^2(R)    
        ###################################################################################################################
        Final = 0; #Datos del final a eliminar (por errores en inconsistencias de tamaño en vecindades)

        lines!(
               Sigma2_Ax, Rango[2:end - Final], (σ2[1:end - Final] ./ Rango[2:end - Final]) .+ 0.3*(Int(floor(NSides/2)) - Tamaño_Tesela),
               color = (Paleta_Colores[Tamaño_Tesela], 1)
              )
        if NSides >= 5
            # --- Líneas verticales con los múltiplos de la longitud de escala
            λ = AproxLambda(NSides);                   #Longitud de escala del primer pico
            vlines!(
                    Sigma2_Ax, [i*λ for i in 1:Int(floor(FN * Radio/λ))],
                    linestyle = :dot,
                    alpha = 1,
                    linewidth = 5,
                    color = :red
                   )
        end
    end
    ###################################################################################################################
    #                                            Guardamos las gráficas    
    ###################################################################################################################
    Fig
end

### Visualización de la $g(R)$

In [ ]:
###################################################################################################################
#                                   Datos del sistema cuasiperiódico 2D-1D
###################################################################################################################
NSides = 5;         #Simetría rotacional del sistema cuasiperiódico en 2D (Antes de la proyección)
Tamaño_Tesela = 1;  #Ranking de la tesela (1 = Más pequeña)
Radio = 500;        #Radio de la "vecindad circular" en 1D (Mitad del tamaño de la región cuadrada centrada en el origen)
###################################################################################################################
#                             Datos de los notebooks para la generación de datos
###################################################################################################################
Notebooks = 20;     #Número de Notebooks empleados
Vecindades = 500;   #Número de Vecindades por Notebook
###################################################################################################################
#                                   Datos para el cálculo de la N(R) y N^2(R)
###################################################################################################################
ΔR = 0.001;     #Tamaño del salto entre R y R

for NSides in 5:2:17
    ###################################################################################################################
    #                                       Factor de normalización de Torquato
    ###################################################################################################################
    Rho = Rho_Dict[NSides]; #Densidad del decorado en vértices, no en centroides
    FN = 2*sqrt(π*Rho);     #Factor de normalización para mantener los resultados independientes de la densidad de puntos
    ###################################################################################################################
    #                                   Definición de las características del lienzo
    ###################################################################################################################
    Radio_Viz = 275;
    if NSides >= 5
        λ = AproxLambda(NSides);            #Longitud de escala del primer pico
        Radio_Viz = Int(ceil(5.1 * (2*λ)));
    end

    Fig = Figure(size = (1800, 800)); #Lienzo en blanco donde se graficara
    gR_Ax = Axis(
                 Fig[1, 1],                                                   #Posición en el lienzo donde se realizará la gráfica
                 title = L"N = %$(NSides)",                                   #Título de la gráfica
                 xlabel = L"R",                                               #Etiqueta que aparece en el eje horizontal
                 ylabel = L"g(R)",                                            #Etiqueta que aparece en el eje vertical
                 titlesize = 75,                                              #Tamaño del título
                 xlabelsize = 75,                                             #Tamaño de la etiqueta al eje horizontal
                 ylabelsize = 75,                                             #Tamaño de la etiqueta al eje vertical
                 xticklabelsize = 60,                                         #Tamaño para el eje X
                 yticklabelsize = 60,                                         #Tamaño para el eje Y
                 xticksize = 45,                                              #Tamaño de los ticks horizontales
                 yticksize = 45,                                              #Tamaño de los ticks verticales
                 limits = ((ΔR, Radio_Viz), nothing),                         #Límites de la visualización para la gráfica
                );
    hidespines!(gR_Ax, :t, :r); #Remueve las líneas de la caja que rodea a la gráfica ':t' = top, ':r' = right
    hidedecorations!(
                     gR_Ax,
                     label = false,           #Se oculta o no las etiquetas a los ejes
                     ticklabels = false,      #Se oculta o no los valores de los ticks de los ejes
                     ticks = false            #Se oculta o no los ticks de los ejes
                    )
    for Tamaño_Tesela in 1:Int(floor(NSides/2))
        ###################################################################################################################
        #                                        Lectura de los datos de g(R)
        ###################################################################################################################
        PesosHist = vec(readdlm(DataPath * "N$(NSides)/A$(Tamaño_Tesela)/Histograma_Pesos_N$(NSides)_CentroidDeco_Area$(Tamaño_Tesela)_Muestra$(Vecindades)_R$(Radio)_S0P001_1.csv"));

        for Nb in 2:Notebooks
            PesosHist .+= vec(readdlm(DataPath * "N$(NSides)/A$(Tamaño_Tesela)/Histograma_Pesos_N$(NSides)_CentroidDeco_Area$(Tamaño_Tesela)_Muestra$(Vecindades)_R$(Radio)_S0P001_$(Nb).csv"));
        end

        #Dividimos entre el número de vecindades acumuladas que se sumaron para obtener los datos
        PesosHist = PesosHist ./ (Vecindades * Notebooks);

        #Densidad numérica de puntos del decorado
        Rho_Num = (sum(PesosHist) + 1) / (π * Radio^2);
        println(Rho_Num)

        #NOTA: La variable "Factor" se compone de dos términos. El primero (1/Δ) es básicamente el inverso del 'Step' empleado en la variación del R para generar los datos originales de los que se obtuvo la altura del histograma
        #contenido en los archivos. El segundo término (1/100) se emplea para "contrarrestar" un factor de '100' que se introduce más adelante a la altura de los histogramas 'PesosHist', este factor de '100' se usa para optimizar 
        #el cálculo de las envolventes con la función ConcaveHull y no se requiere para ninguna otra función.
        Factor = (1/ΔR)/100;

        #NOTA: El Factor de Normalización que introduce Torquato en su artículo sobre Hiperuniformidad (aparece en el Suplemento) toma la forma 'Factor_Normalizacion = 1/(2*sqrt(π*Rho))'. El inverso de este Factor es el que se requiere 
        #al multiplicar las distancias, motivo por el que aparece dicha expresión en la normalización del Rango.
        #NOTA 2: La normalización de la frecuencia se realiza con los valores de R y ΔR antes del factor introducido por Torquato porque es una normalización al histograma que se obtuvo antes del factor de Torquato.
        Rango = 0.0:ΔR:Radio;                                                   #Intervalo de radios empleados
        Rango = [(Rango[i+1] + Rango[i])/2 for i in 1:(length(Rango) - 1)];     #Arreglo con los valores centrales de cada bin del histograma
        RangoNorm = FN .* Rango;                                                #Re-escalamos las longitudes con el factor de Torquato
        Frecuencia = 100*(PesosHist .+ 1e-6)./(2*π*Rho_Num*Rango);              #Frecuencias de los histogramas normalizados
        ###################################################################################################################
        #                                            Gráfica de la g(R)    
        ###################################################################################################################
        Paleta_Colores = cgrad(:inferno, Int(floor(NSides/2)) + 1, categorical = true); #Definimos el gradiente de colores para la gráfica

        ###VISUALIZACIÓN DE LOS DATOS
        Inicio0 = 1;                                                        #Punto inicial a partir del cual se grafica.
        if NSides > 7
            AlturasY = Factor .* Frecuencia;                                #Alturas del histograma tras normalizarse para que tiendan a 1.
            Inicio0 = findfirst(x -> x == maximum(AlturasY), AlturasY) + 1; #Índice del primer dato posterior al pico de varios órdenes de magnitud mayor.
        end
        Final0 = 0;                                                         #Punto final en el cual se deja de graficar

        # --- Gráfica de los datos de la g(R) ---
        lines!(
               gR_Ax, RangoNorm[Inicio0:(end - Final0)], (YOffSet_Dict[NSides]*(Int(floor(NSides/2)) - Tamaño_Tesela)) .+ (Factor .* Frecuencia[Inicio0:(end - Final0)]),
               color = (Paleta_Colores[Tamaño_Tesela], 1)
              )
        
        if NSides >= 5
            # --- Líneas verticales con los múltiplos de la longitud de escala λ_N
            λ = AproxLambda(NSides);    #Longitud de escala del primer pico
            vlines!(
                    gR_Ax, [i*(2*λ) for i in 1:Int(floor(FN * Radio/(2*λ)))],
                    linestyle = :dot,
                    alpha = 1,
                    linewidth = 5,
                    color = :red
                   )
            # --- Líneas verticales con los múltiplos de la longitud de escala κ ---
            κ_N = (NSides/4)*FN;
            vlines!(
                    gR_Ax, [i*(2*κ_N) for i in 1:Int(floor((2*λ)/(2*κ_N)))],
                    linestyle = :dot,
                    alpha = 1,
                    linewidth = 5,
                    color = :black
                   )
        end
    end
    ###################################################################################################################
    #                                            Guardamos las gráficas    
    ###################################################################################################################
    Fig
end